# Forward Process → Prior Check

**Goal**: Verify that the forward diffusion process converges to the SDE prior at $t = T$.

At the end of the forward process every noised sample $x_T$ should be a draw from the theoretical prior $p_T(x)$.
If they match the SDE is correctly implemented and the model has a well-posed training target.

### What we compare
| SDE | Prior $p_T$ | Expected |
|-----|-------------|----------|
| **VE / GBM** | $\mathcal{N}(x_0,\, \sigma_{\max}^2 I)$ — mean stays at $x_0$, noise explodes | $(x_T - x_0) / \sigma_{\max} \sim \mathcal{N}(0, I)$ |
| **VP / sub-VP** | $\mathcal{N}(0,\, I)$ — signal is shrunk to zero | $x_T \sim \mathcal{N}(0, I)$ |

## 0  Configuration

In [180]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────
FOLDER_PATH  = "../data/replication_returns_other_gbm/"          # folder that contains the CSVs
FOLDER = "replication_returns_other_gbm"  # name of the folder (for saving results)
CONFIG_PATH  = "../configs/replication.yaml"  # YAML with process / train / model blocks
VALUE_COL    = "log_adj_close"                # column to use from each CSV
MAX_FILES    = None   # set an int (e.g. 20) to cap how many CSVs are loaded; None = all
N_WINDOWS    = 2000   # maximum number of windows to sample (for speed)
DEVICE       = "cpu"  # "cuda" if you have a GPU
# ──────────────────────────────────────────────────────────────────────────────

## 1  Imports

In [181]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [182]:
import sys, os, glob, math, random
sys.path.insert(0, os.path.abspath(".."))  # project root on path

import yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats as sp_stats

from src.utils.WIP_processes import Diffusion_Processes

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Imports OK")

Imports OK


## 2  Load Config

In [183]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

train_cfg   = cfg["train"]
process_cfg = cfg["process"]

SEQ_LEN   = int(train_cfg["seq_len"])
STRIDE    = int(train_cfg["stride"])
SDE_TYPE  = process_cfg["sde_type"].lower()
SIGMA_MAX = float(process_cfg.get("sigma_max", 1.0))
BETA_MAX  = float(process_cfg.get("beta_max", 1.0))
EPS_TIME  = float(process_cfg.get("eps", 1e-3))

print(f"SDE type   : {SDE_TYPE}")
print(f"sigma_max  : {SIGMA_MAX}")
print(f"beta_max   : {BETA_MAX}")
print(f"seq_len    : {SEQ_LEN}")
print(f"stride     : {STRIDE}")
print(f"eps_time   : {EPS_TIME}")

SDE type   : gbm
sigma_max  : 10.0
beta_max   : 10.0
seq_len    : 2048
stride     : 400
eps_time   : 0.01


## 3  Build Diffusion Process

In [184]:
diffusion = Diffusion_Processes(process_cfg)
sde       = diffusion.sde
T_end     = sde.T

print(f"SDE class  : {sde.__class__.__name__}")
print(f"T (end time): {T_end}")

# Determine the prior family for labelling plots
if SDE_TYPE in ("ve", "gbm"):
    PRIOR_LABEL = f"[σ_max={SIGMA_MAX}]"
    # We will standardise residuals: (x_T - x0) / sigma_max ~ N(0,1)
    STANDARDISE = True
else:
    PRIOR_LABEL = "N(0,\\, I)"
    # For VP / sub-VP the signal is shrunk; x_T itself should be N(0,1)
    STANDARDISE = False

print(f"Prior      : {PRIOR_LABEL}")
print(f"Standardise: {STANDARDISE}  (True → compare (x_T - x0)/σ_max vs N(0,1))")

SDE class  : GBMLogSDE
T (end time): 1.0
Prior      : [σ_max=10.0]
Standardise: True  (True → compare (x_T - x0)/σ_max vs N(0,1))


## 4  Load CSV Data → Windowed Tensor

In [185]:
csv_files = sorted(glob.glob(os.path.join(FOLDER_PATH, "*.csv")))
if MAX_FILES is not None:
    csv_files = csv_files[:MAX_FILES]
print(f"Found {len(csv_files)} CSV files")

all_windows = []   # list of np arrays shape (seq_len,)

for fpath in csv_files:
    try:
        df  = pd.read_csv(fpath, parse_dates=["date"], dayfirst=True)
        ser = df[VALUE_COL].dropna().values.astype(np.float32)
    except Exception as e:
        print(f"  [skip] {os.path.basename(fpath)}: {e}")
        continue

    # Sliding windows
    starts = range(0, max(1, len(ser) - SEQ_LEN + 1), STRIDE)
    for s in starts:
        window = ser[s : s + SEQ_LEN]
        if len(window) == SEQ_LEN:
            all_windows.append(window)

print(f"Total windows (before cap): {len(all_windows)}")

# Cap and shuffle
if N_WINDOWS is not None and len(all_windows) > N_WINDOWS:
    random.shuffle(all_windows)
    all_windows = all_windows[:N_WINDOWS]

# Shape: (B, K=1, L)
x0 = torch.tensor(np.stack(all_windows)[:, None, :], dtype=torch.float32)  # (B, 1, L)
print(f"Tensor x0 shape: {x0.shape}  (B, K, L)")
print(f"  mean={x0.mean():.4f}  std={x0.std():.4f}  min={x0.min():.4f}  max={x0.max():.4f}")

Found 210 CSV files
Total windows (before cap): 5568
Tensor x0 shape: torch.Size([2000, 1, 2048])  (B, K, L)
  mean=2.9104  std=1.8633  min=-4.1366  max=10.0799


## 5  Forward Process at Multiple Times

We apply the forward process at a grid of fixed times $t \in \{0.25,\, 0.5,\, 0.75,\, 1.0\} \cdot T$ and visualise how the marginal distribution evolves toward the prior.

In [186]:
x0_dev = x0.to(DEVICE)
B = x0_dev.shape[0]

time_fractions = [0.25, 0.5, 0.75, 1.0]
t_values = [frac * T_end for frac in time_fractions]

noised_results = {}   # t_val -> (x_t, eps, std)
print(f"t=0.000  std(x0)={x0_dev.std():.4f}  mean(x0)={x0_dev.mean():.4f}")

for t_val in t_values:
    t_tensor = torch.full((B,), t_val, device=DEVICE, dtype=torch.float32)
    x_t, _, eps, std = diffusion.forward_process(x0_dev, t=t_tensor)
    noised_results[t_val] = (x_t.cpu(), eps.cpu(), std.cpu())
    print(f"t={t_val:.3f}  std(x_t)={x_t.std():.4f}  mean(x_t)={x_t.mean():.4f}")

t=0.000  std(x0)=1.8633  mean(x0)=2.9104
t=0.250  std(x_t)=1.8899  mean(x_t)=2.9102
t=0.500  std(x_t)=2.1142  mean(x_t)=2.9109
t=0.750  std(x_t)=3.6712  mean(x_t)=2.9121
t=1.000  std(x_t)=10.1685  mean(x_t)=2.9098


## 6  Prior Samples for Comparison

In [187]:
# Draw the same number of points from the theoretical prior
prior_samples = sde.prior_sampling(x0.shape).cpu()   # (B, 1, L)
print(f"Prior samples shape: {prior_samples.shape}")
print(f"  mean={prior_samples.mean():.4f}  std={prior_samples.std():.4f}")

Prior samples shape: torch.Size([2000, 1, 2048])
  mean=0.0038  std=9.9975


## 7  Marginal Distribution Across Time (Histograms + KDE)

In [188]:
n_cols = len(time_fractions) + 1   # +1 for the prior
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), sharey=False)

xs_ref = np.linspace(-4, 4, 300)
ref_pdf = sp_stats.norm.pdf(xs_ref)

for ax, t_val, frac in zip(axes[:-1], t_values, time_fractions):
    x_t_flat = noised_results[t_val][0].flatten().numpy()   # raw x_t values

    if STANDARDISE:
        # VE / GBM: normalise by sigma_max (constant) — NOT by sigma(t)
        # This tests whether the noise has washed out the signal yet.
        # At t=0 → narrow peak around x0/sigma_max
        # At t=T → should match N(0,1)
        plot_vals = x_t_flat / SIGMA_MAX
        xlabel = r"$x_t \;/\; \sigma_{\max}$"
    else:
        # VP / sub-VP: x_T itself should converge to N(0, I)
        plot_vals = x_t_flat
        xlabel = r"$x_t$"

    ax.hist(plot_vals, bins=80, density=True, alpha=0.6,
            color="steelblue", label=f"noised t={frac}T")
    ax.plot(xs_ref, ref_pdf, "r--", lw=1.5, label=r"$\mathcal{N}(0,1)$")

    ax.set_title(f"$t = {frac}T$")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density" if ax is axes[0] else "")
    ax.legend(fontsize=7)
    ax.grid(True, linewidth=0.4)

# ── Prior panel (rightmost) ───────────────────────────────────────────────────
ax_prior = axes[-1]
prior_flat = prior_samples.flatten().numpy()   # shape (B*1*L,)

if STANDARDISE:
    # prior_sampling() returns randn() * sigma_max  →  divide by sigma_max
    prior_normalised = prior_flat / SIGMA_MAX   # should be N(0,1)
    ax_prior.hist(prior_normalised, bins=80, density=True, alpha=0.6,
                  color="darkorange", label=r"prior / $\sigma_{\max}$")
    ax_prior.plot(xs_ref, ref_pdf, "r--", lw=1.5, label=r"$\mathcal{N}(0,1)$")
    ax_prior.set_xlabel(r"prior $/ \;\sigma_{\max}$")
else:
    ax_prior.hist(prior_flat, bins=80, density=True, alpha=0.6,
                  color="darkorange", label="prior samples")
    ax_prior.plot(xs_ref, ref_pdf, "r--", lw=1.5, label=r"$\mathcal{N}(0,1)$")
    ax_prior.set_xlabel("value")

ax_prior.set_title("Prior $p_T$")
ax_prior.legend(fontsize=7)
ax_prior.grid(True, linewidth=0.4)

fig.suptitle(
    f"Forward process evolution — {sde.__class__.__name__}\n"
    f"({'VE/GBM: plot $x_t/\\sigma_{{max}}$' if STANDARDISE else 'VP: plot $x_t$'}, "
    f"σ_max={SIGMA_MAX})",
    fontsize=11
)
plt.tight_layout()
plt.show()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27512\3057754911.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The distribution is centered at approx. 0 also for t=T, because when we take x_t and we divide it by sigma_max then we get that 2.00/50 is approximately 0.04, therefore the difference is not really visible.

## 8  Prior Match at $t = T$: Detailed Comparison

We focus on the final time $t = T$ and compare the empirical distribution of noised data
with the analytical prior using four complementary views:
1. Histogram / KDE overlay
2. Q–Q plot against $\mathcal{N}(0, 1)$
3. Empirical CDF vs analytical CDF
4. Standard deviation and mean as functions of $t$ (noise growth curve)

In [189]:
x_T, eps_T, std_T = noised_results[T_end]

if STANDARDISE:
    # Correct check: x_T / sigma_max should be N(0,1) if noise dominates signal
    # (NOT (x_T - x0)/sigma_T which is trivially N(0,1) by construction)
    z_noised = x_T.flatten().numpy() / SIGMA_MAX
    compare_label = r"$x_T \;/\; \sigma_{\max}$"
    sigma_T_scalar = std_T.mean().item()
    note = f"σ(T) = {sigma_T_scalar:.4f}  (should equal σ_max = {SIGMA_MAX})"
else:
    z_noised = x_T.flatten().numpy()
    compare_label = r"$x_T$"
    sigma_T_scalar = std_T.mean().item()
    note = f"σ(T) = {sigma_T_scalar:.4f}  (should be close to 1)"

z_prior = sde.prior_sampling(x0.shape).flatten().numpy()
if STANDARDISE:
    z_prior = z_prior / SIGMA_MAX   # prior = randn()*sigma_max → divide → N(0,1)

print(note)
print(f"Noised : mean={z_noised.mean():.4f}  std={z_noised.std():.4f}  n={z_noised.size:,}")
print(f"Prior  : mean={z_prior.mean():.4f}  std={z_prior.std():.4f}  n={z_prior.size:,}")


σ(T) = 10.0000  (should equal σ_max = 10.0)
Noised : mean=0.2910  std=1.0168  n=4,096,000
Prior  : mean=0.0000  std=0.9998  n=4,096,000


## 9  Statistical Tests

In [190]:
# --- KS test: noised @ t=T vs N(0,1) ---
ks_stat, ks_p = sp_stats.kstest(z_noised, "norm")

# --- KS test: prior samples vs N(0,1) ---
ks_stat_prior, ks_p_prior = sp_stats.kstest(z_prior, "norm")

# --- Two-sample KS: noised vs prior ---
ks2_stat, ks2_p = sp_stats.ks_2samp(z_noised, z_prior)

# --- Moment summary ---
def moments(arr, name):
    return {
        "sample": name,
        "n":      len(arr),
        "mean":   arr.mean(),
        "std":    arr.std(),
        "skew":   sp_stats.skew(arr),
        "kurt (excess)": sp_stats.kurtosis(arr),
    }

summary = pd.DataFrame([
    moments(z_noised, f"noised @ t=T"),
    moments(z_prior,  "prior samples"),
    moments(np.random.randn(len(z_noised)), "N(0,1) reference"),
]).set_index("sample")

print("=" * 65)
print(f"SDE: {sde.__class__.__name__}  |  SDE type: {SDE_TYPE.upper()}")
print("=" * 65)
print()
print("Moment summary:")
print(summary.to_string(float_format="{:.4f}".format))
print()
print("Kolmogorov–Smirnov tests:")
print(f"  noised vs N(0,1)  : D={ks_stat:.4f}  p={ks_p:.4e}")
print(f"  prior  vs N(0,1)  : D={ks_stat_prior:.4f}  p={ks_p_prior:.4e}")
print(f"  noised vs prior   : D={ks2_stat:.4f}  p={ks2_p:.4e}")
print()
print("Interpretation:")
print("  A large p-value (> 0.05) means we cannot reject the null that the")
print("  samples come from N(0,1). A small D-statistic confirms the match.")

SDE: GBMLogSDE  |  SDE type: GBM

Moment summary:
                        n    mean    std    skew  kurt (excess)
sample                                                         
noised @ t=T      4096000  0.2910 1.0168  0.0028        -0.0032
prior samples     4096000  0.0000 0.9998 -0.0001         0.0005
N(0,1) reference  4096000 -0.0003 1.0001 -0.0008        -0.0015

Kolmogorov–Smirnov tests:
  noised vs N(0,1)  : D=0.1149  p=0.0000e+00
  prior  vs N(0,1)  : D=0.0002  p=9.6395e-01
  noised vs prior   : D=0.1149  p=0.0000e+00

Interpretation:
  A large p-value (> 0.05) means we cannot reject the null that the
  samples come from N(0,1). A small D-statistic confirms the match.


## 10  Per-Feature Noise Growth Check

For each feature dimension $k$ independently, we check that the empirical std of the noised residual at $t=T$ matches $\sigma_{\max}$.

In [191]:
K = x0.shape[1]   # number of features
t_T_tensor = torch.full((B,), T_end, dtype=torch.float32)
x_T_all, _, _, std_T_all = diffusion.forward_process(x0_dev, t=t_T_tensor)
x_T_all = x_T_all.cpu()

rows = []
for k in range(K):
    vals = x_T_all[:, k, :].flatten().numpy()   # (B*L,)
    x0_k = x0[:, k, :].flatten().numpy()

    if STANDARDISE:
        sigma_T_k = std_T_all.mean().item()  # scalar (same across features for VE)
        z_k = (vals) / (sigma_T_k + 1e-8)
    else:
        z_k = vals

    ks_k, p_k = sp_stats.kstest(z_k, "norm")
    rows.append({
        "feature": k,
        "mean(z)": z_k.mean(),
        "std(z)":  z_k.std(),
        "skew":    sp_stats.skew(z_k),
        "kurt (excess)": sp_stats.kurtosis(z_k),
        "KS D":   ks_k,
        "KS p":   p_k,
    })

df_feat = pd.DataFrame(rows).set_index("feature")
print(df_feat.to_string(float_format="{:.4f}".format))

         mean(z)  std(z)    skew  kurt (excess)   KS D   KS p
feature                                                      
0         0.2912  1.0172 -0.0003        -0.0029 0.1152 0.0000


## 11  Noise eps ~ N(0,1) Self-Check

As a sanity check, the raw noise $\varepsilon$ drawn inside `forward_process` should itself be $\mathcal{N}(0, I)$ — entirely independent of the SDE.

In [192]:
eps_flat = eps_T.flatten().numpy()   # the noise returned at t=T

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(eps_flat, bins=100, density=True, alpha=0.6, color="purple", label="ε samples")
xs = np.linspace(-4, 4, 300)
axes[0].plot(xs, sp_stats.norm.pdf(xs), "r--", lw=1.5, label="$\\mathcal{N}(0,1)$")
axes[0].set_title("Raw noise $\\varepsilon \\sim \\mathcal{N}(0,I)$ check")
axes[0].set_xlabel("ε")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
axes[0].grid(True, linewidth=0.4)

eps_sub  = np.sort(np.random.choice(eps_flat, min(5000, len(eps_flat)), replace=False))
theo_eps = sp_stats.norm.ppf(np.linspace(0.001, 0.999, len(eps_sub)))
axes[1].scatter(theo_eps, eps_sub, s=1, alpha=0.3, color="purple")
axes[1].plot([-4, 4], [-4, 4], "r--", lw=1.5)
axes[1].set_xlabel("$\\mathcal{N}(0,1)$ quantiles")
axes[1].set_ylabel("$\\varepsilon$ quantiles")
axes[1].set_title("Q–Q: noise $\\varepsilon$")
axes[1].grid(True, linewidth=0.4)

ks_eps, p_eps = sp_stats.kstest(eps_flat, "norm")
print(f"ε vs N(0,1): KS D={ks_eps:.4f}  p={p_eps:.4e}")
print(f"  mean={eps_flat.mean():.4f}  std={eps_flat.std():.4f}")

plt.tight_layout()
plt.show()

ε vs N(0,1): KS D=0.0003  p=8.9869e-01
  mean=-0.0001  std=0.9995


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27512\742995870.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ACF

In [ ]:
## 13  Stylized Facts: ACF at Multiple Noise Levels

import sys
sys.path.insert(0, os.path.abspath(".."))
import replication.stylized_facts as sf
from pathlib import Path

output_dir = Path("../images/forward_stylized_facts")
output_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Reference windows as object array of 1-D numpy arrays ─────────────────
# x0 shape: (B, 1, L) — already contains log-returns
all_windows_np = x0[:, 0, :].numpy()  # (B, L)

ref_paths_obj = np.empty(len(all_windows_np), dtype=object)
for i, r in enumerate(all_windows_np):
    ref_paths_obj[i] = r

# ── 2. ACF at multiple noise levels ──────────────────────────────────────────
noise_levels = [0.01, 0.25, 0.50, 0.75, 1.0]

for t_frac in noise_levels:
    t_val   = t_frac * T_end
    t_label = f"t{str(t_frac).replace('.', '')}"   # e.g. "t025"

    if t_frac == 0.01:
        # Near-zero: use original data (no forward process at t≈0)
        x_t_np = all_windows_np
    else:
        t_tensor = torch.full((B,), t_val, dtype=torch.float32)
        x_t, _, _, _ = diffusion.forward_process(x0_dev, t=t_tensor)
        x_t_np = x_t.cpu()[:, 0, :].numpy()   # (B, L)

    paths_obj = np.empty(len(x_t_np), dtype=object)
    for i, r in enumerate(x_t_np):
        paths_obj[i] = r

    sf.acf(
        paths_obj,
        file_name=str(output_dir / f"acf_noised_{t_label}"),
        for_abs=True,
        multiple=True,
        fit=False,
        scale="log",
        max_lag=1000,
    )
    print(f"Saved ACF plot for t={t_frac} → {output_dir}/acf_noised_{t_label}_{FOLDER}")

# ── 3. Reference ACF (clean data) for baseline comparison ────────────────────
sf.acf(
    ref_paths_obj,
    file_name=str(output_dir / f"acf_reference_t0_{FOLDER}"),
    for_abs=True,
    multiple=True,
    fit=False,
    scale="log",
    max_lag=1000,
)
print(f"Saved reference ACF → {output_dir}/acf_reference_t0")


Saved ACF plot for t=0.01 → ..\images\forward_stylized_facts/acf_noised_t001_replication_returns_other_gbm
Saved ACF plot for t=0.25 → ..\images\forward_stylized_facts/acf_noised_t025_replication_returns_other_gbm
Saved ACF plot for t=0.5 → ..\images\forward_stylized_facts/acf_noised_t05_replication_returns_other_gbm


## 12  Summary

| Check | Pass criterion | Typical failure |
|-------|---------------|-----------------|
| **Histogram / KDE** | Noised & prior histograms overlap the $\mathcal{N}(0,1)$ curve | Systematic offset → mean of data ≫ 0; heavy tails → fat-tailed data not centred |
| **Q–Q plot** | Points fall on the $y=x$ diagonal | Curved away at tails → heavier/lighter tails than Gaussian |
| **ECDF** | ECDF tracks $\Phi(z)$ closely | Horizontal gap in the middle → scale mismatch |
| **Noise growth** | Empirical std$(x_t)$ matches $\sigma(t)$ curve | Dots above/below curve → σ schedule tuning needed |
| **KS test** | Large $p$-value (> 0.05) or small $D$ | Small $p$ → distributional mismatch at $t=T$ |
| **ε check** | $\varepsilon$ is exactly $\mathcal{N}(0,1)$ | Should always pass — any failure indicates a bug |

### Key insight for VE / GBM SDEs
The prior is $\mathcal{N}(x_0, \sigma_{\max}^2 I)$, **not** $\mathcal{N}(0, \sigma_{\max}^2 I)$.
If $\sigma_{\max}$ is comparable to the typical magnitude of $x_0$, the noised data at $t=T$ will still carry
a signal from $x_0$ (non-zero mean shift).  
This is intentional for VE: the denoising score model must learn to remove additive Gaussian noise,
not to shrink a signal to zero.